<p align="center"><img src="https://raw.githubusercontent.com/ComputationalChem-USC/AutoMeKin-X/main/banner.jpg" alt="AutoMeKin-X" width="600"></p>

# AutoMeKin-X in Colab: formic acid with PM7 and MACE

This notebook runs a complete, small **AutoMeKin-X** example:

1. **Low level (LL):** reaction discovery for formic acid (FA) with MD simulations at the **PM7** semiempirical level (MOPAC).
2. **High level (HL):** refinement of the LL transition states, minima and products with the **MACE** machine-learning potential (`MACE-omol-0-extra-large-1024`), instead of a quantum chemistry code.
3. **Analysis:** kinetics and interactive reaction networks with [`amk_tools_X`](https://github.com/OmarRodriguezLopez1398/amk_tools_X), after the low level and after MACE.

**Runtime:** go to *Runtime → Change runtime type* and select a **GPU** (a T4 is enough). MACE also runs on CPU, but more slowly.
With the default molecule the whole notebook takes about **15-20 minutes** (installation ~5 min, low level ~4 min, MACE ~5-10 min).

Documentation: [https://computationalchem-usc.github.io/AutoMeKin-X](https://computationalchem-usc.github.io/AutoMeKin-X) · Source code: [https://github.com/ComputationalChem-USC/AutoMeKin-X](https://github.com/ComputationalChem-USC/AutoMeKin-X)

# 1. Installing system packages

In [ ]:
%%capture
!sudo apt-get update -y
!sudo apt-get install -y bc gawk gcc gfortran parallel sqlite3 libsqlite3-dev zip
!pip install ase pubchempy

# 2. Installing AutoMeKin-X

In [ ]:
%%capture
%cd /content/
!git clone https://github.com/ComputationalChem-USC/AutoMeKin-X.git
%cd /content/AutoMeKin-X
!autoreconf -i
!./configure --prefix=/opt/AutoMeKin
!make && make install
# Use all the available processors
!sed -i 's@ignore=1@ignore=0@g' /opt/AutoMeKin/bin/utils.sh
%env PATH=.:/opt/amk_tools_X/scripts:/opt/AutoMeKin/bin:/opt/AutoMeKin/bin/HLscripts:/opt/AutoMeKin/bin/MOPAC_DEV:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
%env LIBRARY_PATH=/opt/AutoMeKin/lib:/opt/AutoMeKin/bin/MOPAC_DEV:/usr/local/cuda/lib64/stubs
%env AMK=/opt/AutoMeKin
%env inter=0
%cd /content
!rm -rf /content/AutoMeKin-X

# 3. Making the input files

The default molecule is **formic acid (HCOOH)**. To study another system, change the fields below and pick it from [PubChem](https://pubchem.ncbi.nlm.nih.gov/) by **name**, **cid**, **smiles**, **inchi**, **inchikey** or **formula**. Larger molecules take much longer.

In [ ]:
#@title Choose the molecule
molecule = "formic acid" #@param {type:"string"}
identifier = "name" #@param ["name", "cid", "smiles", "inchi", "inchikey", "formula"]

%cd /content
!rm -rf test && mkdir test
%cd /content/test

import pubchempy as pcp

# 3D structure from PubChem
m = pcp.get_compounds(molecule, identifier, record_type='3d')[0]
with open('mol.xyz', 'w') as f:
    f.write(str(len(m.atoms)) + '\n\n')
    for a in m.atoms:
        f.write(f"{a.element} {a.x:.6f} {a.y:.6f} {a.z:.6f}\n")

# Input file: AutoMeKin-X's formic acid example, with MACE as the high-level engine
!curl -sL https://raw.githubusercontent.com/ComputationalChem-USC/AutoMeKin-X/main/examples/FA.dat -o mol.dat
!sed -i 's@FA@mol@;s@imagmin 200@imagmin 50@;s@^HighLevel.*@HighLevel mlip mace@;/^IRCpoints/d' mol.dat
from IPython.display import display, Markdown
display(Markdown('**xyz file:**'))
print(open('mol.xyz').read())
print()
display(Markdown('**dat file:**'))
print(open('mol.dat').read())

# 4. Low-level calculations (PM7)

Three iterations of the low-level workflow with 5 tasks per iteration. The input file and every step are explained in the [tutorial](https://computationalchem-usc.github.io/AutoMeKin-X/docs/tutorial.html).

In [ ]:
!llcalcs.sh mol.dat 5 3 2

*Transition states found at the low level*

In [ ]:
!tsll_view.sh

*Convergence table*

In [ ]:
!cat FINAL_LL_mol/convergence.txt

*Coarse-grained reaction network*

In [ ]:
!cat FINAL_LL_mol/RXNet.cg

*Transition states and minima*

In [ ]:
!awk 'BEGINFILE {print "\n"}{print}' FINAL_LL_mol/TSinfo FINAL_LL_mol/MINinfo

*Kinetics at the low level: populations vs time*

In [ ]:
%cd /content/test

import pandas as pd
from matplotlib import pyplot as plt

def plot_kinetics(folder, title):
    data = pd.read_csv(f'{folder}/kinetics.csv')
    x = data.columns[0]
    fig, ax = plt.subplots(figsize=(8, 5))
    for col in data.columns[1:]:
        ax.plot(data[x], data[col], label=col, linewidth=1.2)
    ax.set_xlabel(x, fontsize=14)
    ax.set_ylabel('Population', fontsize=14)
    ax.set_title(title, fontsize=15)
    ax.set_xlim(0, data[x].max())
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

plot_kinetics('FINAL_LL_mol', 'Low level (PM7)')

# 5. Visualizing the low-level network with `amk_tools_X`

Before refining with MACE, have a look at what the low level found. [`amk_tools_X`](https://github.com/OmarRodriguezLopez1398/amk_tools_X) turns the `FINAL_LL_mol` folder into an interactive dashboard (an HTML file) with the reaction network, the 3D structures, the normal modes and the energy profiles. All its features are explained in section 7.

In [ ]:
#@title Installing `amk_tools_X`
%%capture
%cd /opt/
!git clone https://github.com/OmarRodriguezLopez1398/amk_tools_X.git
%cd /opt/amk_tools_X
# Plain install (not -e): the modules go to site-packages, so this running session can import them
!pip install .
%cd /content/test

In [ ]:
#@title Helper to show a dashboard inside the notebook
import subprocess
from google.colab import output

def show_dashboard(html_file, height=850):
    """Serve /content/test/HTML and display one of its dashboards."""
    subprocess.run(['fuser', '-k', '8000/tcp'], stderr=subprocess.DEVNULL)
    subprocess.Popen(['python', '-m', 'http.server', '8000'], cwd='/content/test/HTML',
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    output.serve_kernel_port_as_iframe(8000, path='/' + html_file, height=height)

In [ ]:
#@title Low-level dashboard
%cd /content/test
!mkdir -p HTML
from os import path
tag = 1
if path.isfile('tsdirLL_mol/KMC/starting_minimum'):
    tag = open('tsdirLL_mol/KMC/starting_minimum').read().split()[0]
!amk_gen_view.py FINAL_LL_mol RXNet.cg --barrierless --paths MIN{tag} --outfile HTML/network_LL.html --title "Low level (PM7)"
show_dashboard('network_LL.html')

# 6. High-level refinement with MACE

The low-level transition states are now re-optimized with the **MACE-OMol** machine-learning interatomic potential, and their IRCs, minima, products and kinetics are recomputed. AutoMeKin-X calls MACE through ASE and [Sella](https://github.com/zadorlab/sella), so no quantum chemistry code is needed. The input file already contains `HighLevel mlip mace` (see section 3).

More details: [Using ML potentials](https://computationalchem-usc.github.io/AutoMeKin-X/docs/mlip.html).

In [ ]:
#@title Installing MACE and downloading the model
%%capture
!pip install sella mace-torch
# PyTorch >= 2.6 needs this to load a file shipped with e3nn (harmless otherwise)
!sed -i "s/'constants.pt'))/'constants.pt'), weights_only=False)/" $(python -c "import e3nn, os; print(os.path.join(os.path.dirname(e3nn.__file__), 'o3', '_wigner.py'))")
!mkdir -p /opt/AutoMeKin/models
!curl -L https://github.com/ACEsuit/mace-foundations/releases/download/mace_omol_0/MACE-omol-0-extra-large-1024.model -o /opt/AutoMeKin/models/MACE-omol-0-extra-large-1024.model

In [ ]:
#@title Checking the device MACE will use
import torch
if torch.cuda.is_available():
    print('GPU found:', torch.cuda.get_device_name(0))
else:
    print('No GPU found: MACE will run on CPU (slower). Consider Runtime -> Change runtime type -> GPU.')

## Launching the high-level calculations

The next cell runs the whole high-level workflow with MACE: TS re-optimization, IRCs, minima and products, frequencies and kinetics (`hlcalcs.sh mol.dat 2` uses 2 parallel tasks).

In [ ]:
%cd /content/test
!hlcalcs.sh mol.dat 2

*High-level (MACE) transition states and minima*

In [ ]:
!awk 'BEGINFILE {print "\n"}{print}' FINAL_ML_mol/TSinfo FINAL_ML_mol/MINinfo

*Coarse-grained reaction network at the MACE level*

In [ ]:
!cat FINAL_ML_mol/RXNet.cg

*Kinetics: low level vs MACE*

In [ ]:
%cd /content/test
plot_kinetics('FINAL_LL_mol', 'Low level (PM7)')
plot_kinetics('FINAL_ML_mol', 'High level (MACE)')

# 7. Exploring the MACE network with `amk_tools_X`

## What the dashboard offers

`amk_gen_view.py` reads a `FINAL_*` folder and writes a self-contained HTML file (no Python server is needed to open it later; JSmol is loaded from the internet). The dashboard has:

- **Interactive network.** Pan and zoom with the mouse. Hovering over a node (minimum `MIN`, product `PR`) or an edge (transition state `TS`) shows its name and energy.
- **3D viewer (JSmol).** Click a node or an edge to see its 3D structure.
- **Normal modes.** For the selected structure, pick a vibrational mode to animate it (for a TS, the imaginary mode shows the reaction coordinate).
- **Coordinates to the clipboard.** The XYZ geometry of the selected structure is copied automatically, ready to paste into an input file.
- **Search by name.** Type a label (for example `TS4` or `MIN3`) to locate it in large networks.
- **Energy profiles.** When paths are requested with `--paths`, the energy profiles along them are shown, and they can be filtered by energy or by the presence of the selected node or edge.

## Options of `amk_gen_view.py`

`amk_gen_view.py FINALDIR RXNFILE [options]`

| Option | What it does |
|---|---|
| `FINALDIR` | Results folder: `FINAL_LL_mol` (low level) or `FINAL_ML_mol` (MACE). |
| `RXNFILE` | Network file: `RXNet` (full), `RXNet.cg` (coarse-grained, equivalent conformers merged) or `RXNet.rel` (only the relevant paths). |
| `--barrierless`, `-b` | Add the barrierless channels from `RXNet.barrless`. |
| `--paths`, `-p [SOURCE] [TARGET]` | Search paths and add the energy profiles. With `SOURCE` and `TARGET`, only the paths connecting them. With only `SOURCE`, all cycles that pass through it. Alone, every path in the network (slow for big networks). Several labels go comma-separated, and products can be given by their formula, as written in `RXNet.cg`. |
| `--cutoff_path`, `-c N` | Maximum number of intermediates in a SOURCE→TARGET path (default 4). |
| `--efilter`, `-e E` | Keep only paths whose highest point is below `E` kcal/mol. |
| `--unreduced`, `-u` | Show the whole network, not only the nodes that belong to the selected paths. |
| `--ref_state`, `-rs LABEL` | Use another species (e.g. `MIN3`) as the zero of energy. |
| `--vibrations`, `-v N` | Number of normal modes per structure (`-1`, the default, means all). |
| `--geomolden`, `-ng` | Read the geometries from the MOLDEN files instead of the databases. |
| `--outfile`, `-o FILE` | Name of the HTML file (default `network.html`). |
| `--title`, `-t TEXT` | Title shown on the dashboard. |
| `--resolution`, `-r W,H` | Size of the dashboard in pixels (default `1400,800`). |
| `--fasterlayout`, `-f` | Use a faster layout (spring) for large networks. |

The same tools can also be used from Python (`RXReader` and `RXVisualizer` modules). The full guide is in the [amk_tools_X repository](https://github.com/OmarRodriguezLopez1398/amk_tools_X).

## Three common examples

These examples work for any molecule. The next cell picks, from the MACE results, the three things they need:

- **Starting minimum**: the minimum AutoMeKin-X starts the kinetics from (your input structure). It is not always `MIN1` nor the global minimum.
- **Main product**: the product that receives most of the kinetic flux (`rxn_kin.txt`).
- **Energy window**: the lowest barrier leading to the main product plus 5 kcal/mol.

You can overwrite any of them by hand (e.g. `start = 'MIN3'`) before running the examples.

In [ ]:
#@title Picking the starting minimum, the main product and an energy window
%cd /content/test
import os
from collections import defaultdict

def starting_minimum(tsdir):
    """Minimum AutoMeKin-X starts the kinetics from (the input structure); MIN1 if not found."""
    f = f'{tsdir}/KMC/starting_minimum'
    return 'MIN' + open(f).read().split()[0] if os.path.isfile(f) else 'MIN1'

def main_product(finaldir):
    """Product that receives the largest share of the kinetic flux (rxn_kin.txt)."""
    flux = defaultdict(float)
    if os.path.isfile(f'{finaldir}/rxn_kin.txt'):
        for line in open(f'{finaldir}/rxn_kin.txt'):
            p = line.split()
            if len(p) == 3 and not p[1].isdigit():
                flux[p[1]] += float(p[2])
    return max(flux, key=flux.get) if flux else None

def lowest_barrier_to(finaldir, product):
    """Lowest TS energy (kcal/mol) among the reactions in RXNet.cg that form this product."""
    best = None
    for line in open(f'{finaldir}/RXNet.cg').readlines()[2:]:
        if 'PR' not in line or ':' not in line:
            continue
        words = line.split(':')[-1].split()
        if words and words[-1] in ('CONN', 'DISCONN'):
            words = words[:-1]
        if ''.join(words) == product:
            e = float(line.split()[1])
            best = e if best is None else min(best, e)
    return best

start = starting_minimum('tsdirHL_mol')
product = main_product('FINAL_ML_mol')
barrier = lowest_barrier_to('FINAL_ML_mol', product) if product else None
emax = round(barrier + 5, 1) if barrier is not None else None

print('Starting minimum :', start)
print('Main product     :', product)
print('Energy window    :', f'{emax} kcal/mol' if emax is not None else 'not available')

**Example 1. All the reaction paths from the starting minimum, including barrierless channels.** The usual first look at a network: every path through the starting minimum, with its energy profile.

In [ ]:
%cd /content/test
!amk_gen_view.py FINAL_ML_mol RXNet.cg --barrierless --paths {start} --outfile HTML/example1.html --title "MACE: all paths from {start}"
show_dashboard('example1.html')

**Example 2. Only the paths that lead to one product.** Give a source and a target; products are named by their formula, as written in `RXNet.cg`. Here, from the starting minimum to the main product. `--cutoff_path` sets how many intermediates a path may have (default 4).

In [ ]:
%cd /content/test
if product:
    !amk_gen_view.py FINAL_ML_mol RXNet.cg --barrierless --paths {start} {product} --outfile HTML/example2.html --title "MACE: {start} to {product}"
    show_dashboard('example2.html')
else:
    print('No product found in FINAL_ML_mol/rxn_kin.txt: set product by hand, e.g. product = "A+B"')

**Example 3. Only the energetically accessible paths.** `--efilter` keeps the paths whose highest point lies below a threshold (kcal/mol), which helps to find the dominant channels in large networks. Here the threshold is the energy window picked above, and `--unreduced` keeps the rest of the network visible for context.

In [ ]:
%cd /content/test
if emax is not None:
    !amk_gen_view.py FINAL_ML_mol RXNet.cg --barrierless --paths {start} --efilter {emax} --unreduced --outfile HTML/example3.html --title "MACE: paths from {start} below {emax} kcal/mol"
    show_dashboard('example3.html')
else:
    print('No energy window available: set emax by hand, e.g. emax = 60')

## Build your own dashboard

The defaults below reproduce the MACE network with the energy profiles from the starting minimum (`source = "auto"`). Any label can be typed instead, e.g. `MIN3`, and `target` can be a minimum or a product formula. Change the fields and run the cell again to try other options.

In [ ]:
#@title Generate a dashboard
level = "MACE" #@param ["MACE", "LL"]
rxnfile = "RXNet.cg" #@param ["RXNet.cg", "RXNet", "RXNet.rel"]
barrierless = True #@param {type:"boolean"}
source = "auto" #@param {type:"string"}
target = "" #@param {type:"string"}
cutoff_path = 4 #@param {type:"integer"}
energy_filter = 0 #@param {type:"number"}
ref_state = "" #@param {type:"string"}
unreduced = False #@param {type:"boolean"}
vibrations = -1 #@param {type:"integer"}
faster_layout = False #@param {type:"boolean"}

import os, shlex
os.chdir('/content/test')
os.makedirs('HTML', exist_ok=True)
folder = {'MACE': 'FINAL_ML_mol', 'LL': 'FINAL_LL_mol'}[level]
outfile = f'network_{level}_custom.html'
cmd = ['amk_gen_view.py', folder, rxnfile, '--outfile', f'HTML/{outfile}',
       '--title', f'{level}: {rxnfile}', '--cutoff_path', str(cutoff_path),
       '--vibrations', str(vibrations)]
if barrierless: cmd.append('--barrierless')
src = source.strip()
if src.lower() == 'auto':
    src = starting_minimum({'MACE': 'tsdirHL_mol', 'LL': 'tsdirLL_mol'}[level])
if src:
    cmd += ['--paths', src] + ([target.strip()] if target.strip() else [])
if energy_filter > 0: cmd += ['--efilter', str(energy_filter)]
if ref_state.strip(): cmd += ['--ref_state', ref_state.strip()]
if unreduced: cmd.append('--unreduced')
if faster_layout: cmd.append('--fasterlayout')
print(' '.join(shlex.quote(c) for c in cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])
show_dashboard(outfile)

*Save the results*

In [ ]:
%cd /content/test
!zip -qr results.zip FINAL_LL_mol FINAL_ML_mol HTML
print('Download /content/test/results.zip from the Files panel to keep the results and the dashboards.')